# Digit Recognizer — *casa Zaccaria* notebook

Goal is **learning**, not chasing the leaderboard: the CNN dojo before real OCR/HTR.
We grow the pipeline version by version, each step with a **pre-declared success
criterion** and a logged **ADOPT / REJECT** verdict.

**House rules applied here**
- The out-of-fold (OOF) score is the *authoritative* signal. The public leaderboard is noise.
- Compare **like-for-like**: single-model OOF vs single-model OOF. Never a bag vs a single model.
- One variable per experiment, so we always know *which* change moved the needle.
- Metric alignment: OOF metric (accuracy) matches the competition metric before we trust anything.
- A valid submission is produced on day one, to de-risk the plumbing early.
- The experiment log lives inside this notebook from the first commit; it is the write-up skeleton.

## 1. Setup

Reproducible seeds and a single config block. On Kaggle, set the accelerator to **GPU**
(Settings → Accelerator) before the CNN sections — the augmented v2 in particular is slow on CPU.

In [ ]:
import os
import random
import numpy as np
import pandas as pd

SEED = 42
N_SPLITS = 5
DATA_DIR = "../input/digit-recognizer"   # Kaggle default mount path

random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
print("Setup done. Seed:", SEED)

## 2. Load data

785 columns: one `label` + 784 flattened pixels (28×28). Pixels 0–255 → scale to [0,1].
Scaling is not "cleaning"; it just helps the optimizer converge.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

y = train["label"].values
X = train.drop(columns="label").values.astype("float32") / 255.0   # (n, 784)
X_test = test.values.astype("float32") / 255.0                     # (m, 784)
print("train:", X.shape, " test:", X_test.shape, " labels:", np.unique(y))

## 3. Quick EDA — *and the validation decision*

Two questions, both of which drive the validation scheme:

1. **Balanced classes?** If yes, plain accuracy is honest and no imbalance machinery is needed.
2. **Independent samples?** In *Trace the Ace* samples shared a `session_id`, forcing
   `GroupKFold`. Here each image is an independent digit — no grouping key — so the correct
   choice is `StratifiedKFold` (stratified on the label to keep each fold's class mix representative).

In [ ]:
import matplotlib.pyplot as plt

counts = pd.Series(y).value_counts().sort_index()
print(counts.to_dict())
print("min/max class share:", round(counts.min()/len(y), 4), "/", round(counts.max()/len(y), 4))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(counts.index, counts.values)
ax[0].set_title("Class balance"); ax[0].set_xlabel("digit"); ax[0].set_ylabel("count")
ax[0].set_xticks(range(10))

grid = np.zeros((28, 28*10))
for d in range(10):
    grid[:, d*28:(d+1)*28] = X[y == d][0].reshape(28, 28)
ax[1].imshow(grid, cmap="gray_r"); ax[1].set_title("One sample per class"); ax[1].axis("off")
plt.tight_layout(); plt.show()

**Decision (logged):** classes are ~balanced (~10% each) → accuracy is fair, no correction.
Samples are independent → `StratifiedKFold(n_splits=5, shuffle=True)`. This *same split object*
is reused across every version so OOF scores are directly comparable version-to-version.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


def write_submission(pred, filename):
    sub = pd.DataFrame({"ImageId": np.arange(1, len(pred) + 1), "Label": pred})
    sub.to_csv(filename, index=False)
    print(f"{filename} written:", len(sub), "rows")
    return sub

## 4. v0 — logistic regression baseline

The floor. A linear model on raw pixels treats pixel (5,5) and (5,6) as unrelated columns —
no notion of *where* things are. Whatever it reaches is the ceiling of the
"an image is 784 scalars" mindset. Everything after must earn its complexity by beating it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score

oof_v0 = cross_val_predict(
    LogisticRegression(max_iter=1000, C=0.1, n_jobs=-1),
    X, y, cv=skf, method="predict", n_jobs=-1,
)
cv_v0 = accuracy_score(y, oof_v0)
print("v0 OOF accuracy:", round(cv_v0, 4))   # recorded: 0.9207

# Emit a valid submission on day one (de-risk the pipeline).
clf = LogisticRegression(max_iter=1000, C=0.1, n_jobs=-1).fit(X, y)
_ = write_submission(clf.predict(X_test), "submission_v0.csv")

## 5. The CNN

We reshape the 784 columns back into a 28×28×1 **image** and let convolutions learn features
via **locality** (small 3×3 filters look at one neighborhood) and **weight sharing** (the same
filter slides everywhere → translation tolerance, reinforced by `MaxPooling`).

`build_cnn` is defined **once** with an `augment` flag. The architecture is identical whether or
not augmentation is on, so v1 vs v2 differ by *augmentation only* — a fair comparison. The
augmentation block is active during training only (like Dropout); at inference it is a no-op, so
every OOF/test prediction is on clean images.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(SEED)

X_img = X.reshape(-1, 28, 28, 1)
X_test_img = X_test.reshape(-1, 28, 28, 1)

# Augmentation covers POSE (position, orientation, scale), not IDENTITY. Modest magnitudes:
# too much rotation blurs 6 vs 9. These are the exact params visualized in chat.
data_aug = keras.Sequential([
    layers.RandomTranslation(0.1, 0.1),   # +/-10% shift (~+/-2.8 px)
    layers.RandomRotation(0.04),          # +/-~14 deg
    layers.RandomZoom(0.1),               # +/-10% scale
], name="data_aug")


def build_cnn(augment=False):
    inputs = keras.Input(shape=(28, 28, 1))
    x = data_aug(inputs) if augment else inputs
    x = layers.Conv2D(32, 3, activation="relu")(x)   # 32 filters slide over the image
    x = layers.MaxPooling2D()(x)                      # keep strongest response per region
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    m = keras.Model(inputs, outputs)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


build_cnn().summary()

### v1 — first CNN (no augmentation, EarlyStopping on `val_accuracy`)

Deliberately minimal so the v0 → v1 delta cleanly isolates *"the model now knows it is
looking at an image."*

**Pre-declared criterion:** beat v0's OOF (0.9207). If not, it is a bug to hunt.

In [ ]:
oof_v1 = np.zeros(len(y), dtype="int64")
test_probs_v1 = np.zeros((len(X_test_img), 10), dtype="float32")
early_acc = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                          restore_best_weights=True)

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    model = build_cnn(augment=False)
    model.fit(X_img[tr], y[tr], validation_data=(X_img[va], y[va]),
              epochs=20, batch_size=128, callbacks=[early_acc], verbose=2)
    oof_v1[va] = model.predict(X_img[va], verbose=0).argmax(1)     # single-model OOF (authoritative)
    test_probs_v1 += model.predict(X_test_img, verbose=0) / N_SPLITS  # 5-model bag (submission only)
    print(f"fold {fold}:", round(accuracy_score(y[va], oof_v1[va]), 4))

cv_v1 = accuracy_score(y, oof_v1)
print("\nv1 OOF accuracy:", round(cv_v1, 4), "| delta vs v0:", round(cv_v1 - cv_v0, 4),
      "|", "PASS" if cv_v1 > cv_v0 else "FAIL")
_ = write_submission(test_probs_v1.argmax(1), "submission_v1.csv")

> **Note on OOF vs LB.** The OOF (0.9735 on our run) is a *single model per fold*; the submission
> is a *5-model bag*. So the LB (0.98025) sitting above the OOF is the bagging lift, not overfitting.
> All ADOPT/REJECT decisions use single-model OOF, compared like-for-like.

### v1 error analysis — measure, don't eyeball

A confusion matrix is **directional** (`i → j` ≠ `j → i`). We rank both the single worst
directional cells and the worst mutually-confusable pairs.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_v1 = confusion_matrix(y, oof_v1)

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ConfusionMatrixDisplay(cm_v1, display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("v1 OOF confusion matrix"); plt.tight_layout(); plt.show()

off = cm_v1.copy(); np.fill_diagonal(off, 0)

pairs = sorted(((i, j, off[i, j]) for i in range(10) for j in range(10) if off[i, j] > 0),
               key=lambda t: t[2], reverse=True)
print("Top directional confusions (true -> pred):")
for i, j, n in pairs[:6]:
    print(f"  {i} -> {j}: {n:>3}  ({n / cm_v1[i].sum() * 100:.2f}% of the {i}s)")

sym = {(i, j): off[i, j] + off[j, i] for i in range(10) for j in range(i + 1, 10)}
print("\nTop confusable pairs (both directions):")
for (i, j), n in sorted(sym.items(), key=lambda t: t[1], reverse=True)[:5]:
    print(f"  {i} <-> {j}: {n:>3}  ({off[i, j]} one way, {off[j, i]} the other)")

## 6. v1b — controlled ablation: EarlyStopping on `val_loss`

**One variable changed vs v1**, nothing else: monitor `val_loss` (smooth) instead of
`val_accuracy` (a staircase metric that plateaus and can stop training too early).
Same architecture, no augmentation. This isolates whether v1 was stopping prematurely.

**Pre-declared criterion:** beat v1's OOF (our run: 0.9735), single-model vs single-model.

In [ ]:
oof_v1b = np.zeros(len(y), dtype="int64")
early_loss = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                           restore_best_weights=True)

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    model = build_cnn(augment=False)
    model.fit(X_img[tr], y[tr], validation_data=(X_img[va], y[va]),
              epochs=25, batch_size=128, callbacks=[early_loss], verbose=2)
    oof_v1b[va] = model.predict(X_img[va], verbose=0).argmax(1)
    print(f"fold {fold}:", round(accuracy_score(y[va], oof_v1b[va]), 4))

cv_v1b = accuracy_score(y, oof_v1b)
print("\nv1b OOF accuracy:", round(cv_v1b, 4), "| delta vs v1:", round(cv_v1b - cv_v1, 4),
      "|", "PASS" if cv_v1b > cv_v1 else "FAIL")

## 7. v2 — data augmentation

Add pose variation (shift / rotate / zoom) so the network learns to ignore *where* and *how big*
a digit is. The base is the better EarlyStopping config from v1b (`val_loss`).

> **Decision rule (one variable at a time):** if v1b **beat** v1, keep `monitor="val_loss"` below —
> then v2's only change vs the adopted base is augmentation. If v1b **did not** beat v1, change the
> monitor back to `"val_accuracy"` here, so augmentation stays the sole new variable.

Augmentation slows convergence, so epochs/patience rise — a coupled consequence of augmentation,
not an independent knob.

**Pre-declared criterion:** beat the current best OOF (v1 = 0.9735), single-model vs single-model.
**Sub-hypothesis (logged before results):** augmentation should reduce 4↔9 and 8↔9 (pose exposes
the loop feature) but leave 2↔7 ~unchanged (glyph ambiguity, not pose).

In [ ]:
oof_v2 = np.zeros(len(y), dtype="int64")
test_probs_v2 = np.zeros((len(X_test_img), 10), dtype="float32")
early_v2 = keras.callbacks.EarlyStopping(monitor="val_loss", patience=6,
                                         restore_best_weights=True)

for fold, (tr, va) in enumerate(skf.split(X_img, y)):
    model = build_cnn(augment=True)
    model.fit(X_img[tr], y[tr], validation_data=(X_img[va], y[va]),
              epochs=40, batch_size=128, callbacks=[early_v2], verbose=2)
    oof_v2[va] = model.predict(X_img[va], verbose=0).argmax(1)     # clean images (aug off at inference)
    test_probs_v2 += model.predict(X_test_img, verbose=0) / N_SPLITS
    print(f"fold {fold}:", round(accuracy_score(y[va], oof_v2[va]), 4))

cv_v2 = accuracy_score(y, oof_v2)
print("\nv2 OOF accuracy:", round(cv_v2, 4), "| delta vs v1:", round(cv_v2 - cv_v1, 4),
      "|", "PASS" if cv_v2 > cv_v1 else "FAIL")
_ = write_submission(test_probs_v2.argmax(1), "submission_v2.csv")

### v1 → v2: did augmentation move the pairs we predicted?

The hypothesis test. Compare confusion matrices, don't eyeball. Blue = fewer errors than v1.

In [ ]:
cm_v2 = confusion_matrix(y, oof_v2)

def pair(cm, i, j):
    return cm[i, j] + cm[j, i]

print("Pre-declared pairs (both directions):")
for i, j in [(2, 7), (7, 9), (4, 9), (8, 9)]:
    b, a = pair(cm_v1, i, j), pair(cm_v2, i, j)
    print(f"  {i}<->{j}: {b} -> {a}  ({a - b:+d})")

diff = (cm_v2 - cm_v1).astype(int); np.fill_diagonal(diff, 0)
m = max(1, np.abs(diff).max())
fig, ax = plt.subplots(figsize=(5.4, 5.4))
im = ax.imshow(diff, cmap="RdBu_r", vmin=-m, vmax=m)
for i in range(10):
    for j in range(10):
        if diff[i, j]:
            ax.text(j, i, f"{diff[i, j]:+d}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Off-diagonal change: v2 - v1  (blue = fewer errors)")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## 8. Experiment log  *(the write-up skeleton)*

Criterion written **before** results; verdict **after**.

| Version | Description | CV (OOF acc) | Pre-declared criterion | Verdict |
|--------:|-------------|:------------:|------------------------|:-------:|
| v0  | Logistic regression on raw pixels | 0.9207 | baseline floor | REFERENCE |
| v1  | Minimal CNN, EarlyStopping on `val_accuracy` | 0.9735 | beat v0 (0.9207) | ADOPT (+0.0528) |
| v1b | v1 with EarlyStopping on `val_loss` (1 var) | *fill* | beat v1 (0.9735) | *fill* |
| v2  | best-ES base + data augmentation | *fill* | beat v1 (0.9735) | *fill* |
| v3  | v2 + BatchNorm / dropout / deeper head | — | beat v2 | *planned* |

**Running notes**
- v0: OOF 0.9207 ≈ LB 0.9205 → pipeline clean, OOF trustworthy.
- v1: OOF 0.9735 (single model) vs LB 0.98025 (5-model bag). Gap = bagging lift, not overfitting.
- v1 worst confusions: 2↔7 (101), 7↔9 (84), 4↔9 (79), 8↔9 (60). 2→7 alone (54) is the single largest cell.
- *(add one line per experiment: what changed, the delta, the verdict.)*

## 9. Next steps

- **v3** — BatchNorm + dropout + a slightly deeper head, once augmentation is settled.
- Light **Optuna** tuning, always with the current best enqueued as an anti-regression trial.
- If 2↔7 refuses to move across v2/v3, that is evidence of *irreducible* glyph ambiguity — a
  plateau diagnosis, not a modeling failure.